# Step 3.5. Further Preprocessing

이 노트북은 팀원이 Step 3까지 수행한 `BankChurners_Final_Preprocessed.csv`를 기반으로,  
Step 4 Clustering과 Step 5 Segment Profiling에 사용할 최종 입력 데이터를 생성한다.

## 목적

1. Clustering input과 Profiling table을 명확히 분리한다.
2. Label-encoded 범주형 변수는 clustering에 사용하지 않고, profiling용 label로 복원한다.
3. 거래 변화율 극단값을 완화하기 위해 clipping을 적용한다.
4. Segment 해석에 사용할 profiling 파생변수와 high-risk signal을 생성한다.
5. 해석 가능한 고객 행동 축을 생성한다.
6. 최종 산출물을 `data/processed/`에 저장한다.

## 입력 파일

- `data/interim/BankChurners_Final_Preprocessed.csv`

## 출력 파일

- `data/processed/BankChurners_cluster_input.csv`
- `data/processed/BankChurners_profile_table.csv`
- `data/processed/BankChurners_axis_features.csv`
- `data/processed/BankChurners_component_features.csv`
- `data/processed/cluster_feature_info.json`
- `data/processed/preprocessing_validation_summary.csv`
- `data/processed/preprocessing_thresholds.json`
- `data/processed/preprocessing_scalers.pkl`

In [ ]:
# ============================================================
# 0. Import & Project Path Setup
# ============================================================

from pathlib import Path
import sys
import json
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


def find_project_root():
    """
    현재 작업 위치가 project root이든 notebooks 폴더이든,
    src/common.py 또는 config/project_config.json을 기준으로 project root를 찾는다.
    """
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)

    for path in candidates:
        if (path / "src" / "common.py").exists() or (path / "config" / "project_config.json").exists():
            return path

    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

try:
    from src.common import read_csv_safely, save_json
except Exception:
    # folder_setup.py로 생성한 src/common.py를 찾지 못하는 경우를 위한 fallback
    def read_csv_safely(path, **kwargs):
        encodings = ("utf-8-sig", "utf-8", "cp949", "euc-kr")
        last_error = None
        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc, **kwargs)
            except UnicodeDecodeError as exc:
                last_error = exc
        raise last_error

    def save_json(data, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

print("Project root:", ROOT)

## 1. 입력/출력 경로 설정

이 노트북은 절대경로를 사용하지 않는다.  
프로젝트 루트 기준 상대경로만 사용한다.

In [ ]:
# ============================================================
# 1. Input / Output Paths
# ============================================================

input_path = ROOT / "data" / "interim" / "BankChurners_Final_Preprocessed.csv"
output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

if not input_path.exists():
    raise FileNotFoundError(
        f"입력 파일이 없습니다: {input_path}\n"
        "팀원이 만든 BankChurners_Final_Preprocessed.csv를 "
        "data/interim/BankChurners_Final_Preprocessed.csv 위치에 배치하세요."
    )

df = read_csv_safely(input_path)

print("Loaded input file:", input_path.relative_to(ROOT))
print("Data shape:", df.shape)
display(df.head())

## 2. 기본 검증

Step 3까지의 결과 파일이 이후 분석에 필요한 필수 컬럼을 갖고 있는지 확인한다.

In [ ]:
# ============================================================
# 2. Basic Validation
# ============================================================

required_columns = [
    "Attrition_Flag",
    "Gender",
    "Education_Level",
    "Marital_Status",
    "Income_Category",
    "Card_Category",
    "Customer_Age",
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Months_Inactive_12_mon",
    "Total_Ct_Chng_Q4_Q1",
    "Total_Amt_Chng_Q4_Q1",
    "Total_Relationship_Count",
    "Months_on_book",
    "Contacts_Count_12_mon",
    "Credit_Limit",
    "Total_Revolving_Bal",
    "Avg_Utilization_Ratio",
]

missing_cols = [col for col in required_columns if col not in df.columns]

if missing_cols:
    raise ValueError(f"필수 컬럼이 누락되었습니다: {missing_cols}")

validation_initial = {
    "n_rows": int(len(df)),
    "n_columns": int(df.shape[1]),
    "missing_values_total": int(df.isnull().sum().sum()),
    "duplicated_rows": int(df.duplicated().sum()),
}

print("필수 컬럼 확인 완료")
print(json.dumps(validation_initial, ensure_ascii=False, indent=4))

display(pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "missing": df.isnull().sum().values
}).head(30))

## 3. Profiling용 label 복원

팀원이 만든 최종 전처리 파일에는 범주형 변수가 Label Encoding된 상태로 들어 있다.  
이 숫자값은 clustering input에 사용하면 안 된다. 대신 Step 5에서 사람이 해석할 수 있도록 label을 복원한다.

주의할 점:

- `Income_Category = 0, 1, 2, ...`는 소득 순서가 아니다.
- Label-encoded 범주형 변수는 거리 기반 clustering에 넣지 않는다.
- 복원된 label은 segment profiling에서 분포를 확인하는 데만 사용한다.

In [ ]:
# ============================================================
# 3. Restore Human-readable Labels for Profiling
# ============================================================

reverse_maps = {
    "Attrition_Label": (
        "Attrition_Flag",
        {0: "Existing Customer", 1: "Attrited Customer"}
    ),
    "Gender_Label": (
        "Gender",
        {0: "F", 1: "M"}
    ),
    "Education_Label": (
        "Education_Level",
        {
            0: "College",
            1: "Doctorate",
            2: "Graduate",
            3: "High School",
            4: "Post-Graduate",
            5: "Uneducated",
            6: "Unknown"
        }
    ),
    "Marital_Label": (
        "Marital_Status",
        {
            0: "Divorced",
            1: "Married",
            2: "Single",
            3: "Unknown"
        }
    ),
    "Income_Label": (
        "Income_Category",
        {
            0: "$120K +",
            1: "$40K - $60K",
            2: "$60K - $80K",
            3: "$80K - $120K",
            4: "Less than $40K",
            5: "Unknown"
        }
    ),
    "Card_Label": (
        "Card_Category",
        {
            0: "Blue",
            1: "Gold",
            2: "Platinum",
            3: "Silver"
        }
    )
}

label_restore_report = []

for new_col, (old_col, mapping) in reverse_maps.items():
    df[new_col] = df[old_col].map(mapping)
    label_restore_report.append({
        "new_label_column": new_col,
        "source_column": old_col,
        "source_unique_values": sorted(df[old_col].dropna().unique().tolist()),
        "n_unmapped": int(df[new_col].isnull().sum())
    })

label_restore_report = pd.DataFrame(label_restore_report)
display(label_restore_report)

if label_restore_report["n_unmapped"].sum() > 0:
    print("주의: 일부 label 복원에 실패했습니다. mapping과 원본 unique value를 확인하세요.")

display(df[list(reverse_maps.keys())].head())

## 4. 거래 변화율 변수 clipping

`Total_Ct_Chng_Q4_Q1`, `Total_Amt_Chng_Q4_Q1`는 일부 극단값이 클러스터 중심을 과도하게 끌어당길 수 있다.  
따라서 고객을 삭제하지 않고, 1%~99% 범위로 winsorization/clipping을 적용한다.

In [ ]:
# ============================================================
# 4. Winsorization / Clipping for Change-rate Variables
# ============================================================

def winsorize_series(s, lower=0.01, upper=0.99):
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi), float(lo), float(hi)


df["Total_Ct_Chng_Q4_Q1_clip"], ct_lo, ct_hi = winsorize_series(df["Total_Ct_Chng_Q4_Q1"])
df["Total_Amt_Chng_Q4_Q1_clip"], amt_lo, amt_hi = winsorize_series(df["Total_Amt_Chng_Q4_Q1"])

preprocessing_thresholds = {
    "Total_Ct_Chng_Q4_Q1_clip": {
        "lower_quantile": 0.01,
        "upper_quantile": 0.99,
        "lower_value": ct_lo,
        "upper_value": ct_hi
    },
    "Total_Amt_Chng_Q4_Q1_clip": {
        "lower_quantile": 0.01,
        "upper_quantile": 0.99,
        "lower_value": amt_lo,
        "upper_value": amt_hi
    }
}

clip_summary = pd.DataFrame({
    "variable": ["Total_Ct_Chng_Q4_Q1", "Total_Amt_Chng_Q4_Q1"],
    "original_min": [df["Total_Ct_Chng_Q4_Q1"].min(), df["Total_Amt_Chng_Q4_Q1"].min()],
    "original_max": [df["Total_Ct_Chng_Q4_Q1"].max(), df["Total_Amt_Chng_Q4_Q1"].max()],
    "clipped_min": [df["Total_Ct_Chng_Q4_Q1_clip"].min(), df["Total_Amt_Chng_Q4_Q1_clip"].min()],
    "clipped_max": [df["Total_Ct_Chng_Q4_Q1_clip"].max(), df["Total_Amt_Chng_Q4_Q1_clip"].max()]
})

display(clip_summary)

## 5. Profiling용 파생변수 생성

아래 변수들은 clustering input이 아니라, cluster가 만들어진 이후 segment를 해석하기 위한 변수다.

In [ ]:
# ============================================================
# 5. Profiling Feature Engineering
# ============================================================

df["Avg_Ticket_Size"] = np.where(
    df["Total_Trans_Ct"] > 0,
    df["Total_Trans_Amt"] / df["Total_Trans_Ct"],
    np.nan
)

df["Avg_Ticket_Size"] = df["Avg_Ticket_Size"].fillna(0)
df["Avg_Ticket_Size_log"] = np.log1p(df["Avg_Ticket_Size"])

df["Inactive_Ratio_12m"] = df["Months_Inactive_12_mon"] / 12
df["Contact_Rate_12m"] = df["Contacts_Count_12_mon"] / 12

df["No_Revolving_Bal_Flag"] = (df["Total_Revolving_Bal"] == 0).astype(int)
df["Low_Relationship_Flag"] = (df["Total_Relationship_Count"] <= 2).astype(int)

low_util_threshold = float(df["Avg_Utilization_Ratio"].quantile(0.20))
df["Low_Utilization_Flag"] = (
    df["Avg_Utilization_Ratio"] <= low_util_threshold
).astype(int)

preprocessing_thresholds["Low_Utilization_Flag"] = {
    "quantile": 0.20,
    "threshold_value": low_util_threshold
}

profiling_feature_preview = [
    "Avg_Ticket_Size",
    "Avg_Ticket_Size_log",
    "Inactive_Ratio_12m",
    "Contact_Rate_12m",
    "No_Revolving_Bal_Flag",
    "Low_Relationship_Flag",
    "Low_Utilization_Flag"
]

display(df[profiling_feature_preview].describe().T)
display(df[profiling_feature_preview].head())

## 6. High-risk signal 생성

High-risk signal은 cluster를 만들 때 사용하지 않는다.  
Step 5에서 도출된 segment가 실제 위험 행동 신호를 많이 포함하는지 사후 검증하는 데 사용한다.

In [ ]:
# ============================================================
# 6. High-risk Signal Construction
# ============================================================

inactive_threshold = float(df["Months_Inactive_12_mon"].quantile(0.80))
low_ct_change_threshold = float(df["Total_Ct_Chng_Q4_Q1"].quantile(0.20))
low_amt_change_threshold = float(df["Total_Amt_Chng_Q4_Q1"].quantile(0.20))
high_contact_threshold = float(df["Contacts_Count_12_mon"].quantile(0.80))

df["High_Inactive_Flag"] = (
    df["Months_Inactive_12_mon"] >= inactive_threshold
).astype(int)

df["Low_Ct_Change_Flag"] = (
    df["Total_Ct_Chng_Q4_Q1"] <= low_ct_change_threshold
).astype(int)

df["Low_Amt_Change_Flag"] = (
    df["Total_Amt_Chng_Q4_Q1"] <= low_amt_change_threshold
).astype(int)

df["High_Contact_Flag"] = (
    df["Contacts_Count_12_mon"] >= high_contact_threshold
).astype(int)

risk_signal_cols = [
    "High_Inactive_Flag",
    "Low_Ct_Change_Flag",
    "Low_Amt_Change_Flag",
    "High_Contact_Flag",
    "Low_Relationship_Flag",
    "Low_Utilization_Flag"
]

df["High_Risk_Signal_Count"] = df[risk_signal_cols].sum(axis=1)

preprocessing_thresholds.update({
    "High_Inactive_Flag": {
        "quantile": 0.80,
        "threshold_value": inactive_threshold
    },
    "Low_Ct_Change_Flag": {
        "quantile": 0.20,
        "threshold_value": low_ct_change_threshold
    },
    "Low_Amt_Change_Flag": {
        "quantile": 0.20,
        "threshold_value": low_amt_change_threshold
    },
    "High_Contact_Flag": {
        "quantile": 0.80,
        "threshold_value": high_contact_threshold
    },
    "High_Risk_Signal_Count": {
        "components": risk_signal_cols
    }
})

risk_summary = df[risk_signal_cols + ["High_Risk_Signal_Count"]].mean().to_frame("mean")
display(risk_summary)

## 7. 해석 가능한 고객 행동 축 생성

raw 변수 10~11개를 그대로 clustering에 넣지 않고, 프로젝트 목적에 맞는 7개의 행동 축으로 재구성한다.

### 표준화가 두 번 등장하는 이유

1. **Component-level standardization**  
   서로 단위가 다른 원변수들을 하나의 축으로 평균내기 전에 표준화한다.  
   예: 거래 횟수와 거래 금액을 같은 비중으로 `Activity_Score`에 반영

2. **Axis-level standardization**  
   생성된 7개 행동 축이 clustering에서 같은 거리 가중치를 갖도록 다시 표준화한다.

In [ ]:
# ============================================================
# 7. Construct Interpretable Behavior Axes
# ============================================================

component_features = pd.DataFrame({
    "Total_Trans_Ct": df["Total_Trans_Ct"],
    "Total_Trans_Amt_log": np.log1p(df["Total_Trans_Amt"]),
    "Months_Inactive_12_mon": df["Months_Inactive_12_mon"],
    "Total_Ct_Chng_Q4_Q1_clip": df["Total_Ct_Chng_Q4_Q1_clip"],
    "Total_Amt_Chng_Q4_Q1_clip": df["Total_Amt_Chng_Q4_Q1_clip"],
    "Total_Relationship_Count": df["Total_Relationship_Count"],
    "Months_on_book": df["Months_on_book"],
    "Contacts_Count_12_mon": df["Contacts_Count_12_mon"],
    "Credit_Limit_log": np.log1p(df["Credit_Limit"]),
    "Avg_Utilization_Ratio": df["Avg_Utilization_Ratio"],
}, index=df.index)

component_scaler = StandardScaler()

component_z = pd.DataFrame(
    component_scaler.fit_transform(component_features),
    columns=component_features.columns,
    index=df.index
)

axis_features = pd.DataFrame(index=df.index)

axis_features["Activity_Score"] = component_z[
    ["Total_Trans_Ct", "Total_Trans_Amt_log"]
].mean(axis=1)

axis_features["Decline_Risk_Score"] = -component_z[
    ["Total_Ct_Chng_Q4_Q1_clip", "Total_Amt_Chng_Q4_Q1_clip"]
].mean(axis=1)

axis_features["Inactivity_Score"] = component_z["Months_Inactive_12_mon"]

axis_features["Relationship_Strength_Score"] = component_z[
    ["Total_Relationship_Count", "Months_on_book"]
].mean(axis=1)

axis_features["Contact_Burden_Score"] = component_z["Contacts_Count_12_mon"]

axis_features["Credit_Capacity_Score"] = component_z["Credit_Limit_log"]

axis_features["Credit_Utilization_Score"] = component_z["Avg_Utilization_Ratio"]

axis_scaler = StandardScaler()

df_cluster_ready = pd.DataFrame(
    axis_scaler.fit_transform(axis_features),
    columns=axis_features.columns,
    index=df.index
)

df_profile_ready = df.copy()

print("component_features:", component_features.shape)
print("axis_features:", axis_features.shape)
print("df_cluster_ready:", df_cluster_ready.shape)
print("df_profile_ready:", df_profile_ready.shape)

display(axis_features.describe().T)
display(df_cluster_ready.describe().T)

## 8. 저장 전 검증

Clustering input에는 이탈 여부, 인구통계 변수, label-encoded 범주형 변수가 들어가면 안 된다.

In [ ]:
# ============================================================
# 8. Final Validation before Saving
# ============================================================

assert len(df_cluster_ready) == len(df_profile_ready), "cluster input과 profile table의 row 수가 다릅니다."
assert df_cluster_ready.isnull().sum().sum() == 0, "df_cluster_ready에 결측치가 있습니다."
assert df_profile_ready.isnull().sum().sum() == 0, "df_profile_ready에 결측치가 있습니다."

forbidden_cols = [
    "Attrition_Flag",
    "Attrition_Label",
    "Gender",
    "Education_Level",
    "Marital_Status",
    "Income_Category",
    "Card_Category",
    "Customer_Age",
    "Dependent_count"
]

included_forbidden = [col for col in forbidden_cols if col in df_cluster_ready.columns]
assert len(included_forbidden) == 0, f"clustering input에 들어가면 안 되는 컬럼 포함: {included_forbidden}"

validation_final = {
    "n_rows_cluster_input": int(len(df_cluster_ready)),
    "n_features_cluster_input": int(df_cluster_ready.shape[1]),
    "n_rows_profile_table": int(len(df_profile_ready)),
    "n_columns_profile_table": int(df_profile_ready.shape[1]),
    "cluster_input_missing_values": int(df_cluster_ready.isnull().sum().sum()),
    "profile_table_missing_values": int(df_profile_ready.isnull().sum().sum()),
    "forbidden_columns_in_cluster_input": included_forbidden,
    "cluster_features": list(df_cluster_ready.columns),
}

print(json.dumps(validation_final, ensure_ascii=False, indent=4))

## 9. 최종 파일 저장

`Row_ID`를 부여하여 clustering input과 profiling table을 안전하게 연결할 수 있도록 한다.

In [ ]:
# ============================================================
# 9. Save Final Outputs
# ============================================================

df_cluster_save = df_cluster_ready.reset_index(drop=True).copy()
df_profile_save = df_profile_ready.reset_index(drop=True).copy()
df_axis_save = axis_features.reset_index(drop=True).copy()
df_component_save = component_features.reset_index(drop=True).copy()

df_cluster_save.insert(0, "Row_ID", np.arange(len(df_cluster_save)))
df_profile_save.insert(0, "Row_ID", np.arange(len(df_profile_save)))
df_axis_save.insert(0, "Row_ID", np.arange(len(df_axis_save)))
df_component_save.insert(0, "Row_ID", np.arange(len(df_component_save)))

cluster_path = output_dir / "BankChurners_cluster_input.csv"
profile_path = output_dir / "BankChurners_profile_table.csv"
axis_path = output_dir / "BankChurners_axis_features.csv"
component_path = output_dir / "BankChurners_component_features.csv"
thresholds_path = output_dir / "preprocessing_thresholds.json"
metadata_path = output_dir / "cluster_feature_info.json"
validation_path = output_dir / "preprocessing_validation_summary.csv"
scaler_path = output_dir / "preprocessing_scalers.pkl"

df_cluster_save.to_csv(cluster_path, index=False, encoding="utf-8-sig")
df_profile_save.to_csv(profile_path, index=False, encoding="utf-8-sig")
df_axis_save.to_csv(axis_path, index=False, encoding="utf-8-sig")
df_component_save.to_csv(component_path, index=False, encoding="utf-8-sig")

save_json(preprocessing_thresholds, thresholds_path)

with open(scaler_path, "wb") as f:
    pickle.dump(
        {
            "component_scaler": component_scaler,
            "axis_scaler": axis_scaler,
            "component_feature_columns": list(component_features.columns),
            "axis_feature_columns": list(axis_features.columns),
            "risk_signal_columns": risk_signal_cols,
            "preprocessing_thresholds": preprocessing_thresholds,
        },
        f
    )

feature_info = {
    "step": "Step 3.5 Further Preprocessing",
    "input_file": str(input_path.relative_to(ROOT)),
    "output_files": {
        "cluster_input": str(cluster_path.relative_to(ROOT)),
        "profile_table": str(profile_path.relative_to(ROOT)),
        "axis_features": str(axis_path.relative_to(ROOT)),
        "component_features": str(component_path.relative_to(ROOT)),
        "thresholds": str(thresholds_path.relative_to(ROOT)),
        "scalers": str(scaler_path.relative_to(ROOT)),
        "validation_summary": str(validation_path.relative_to(ROOT)),
    },
    "cluster_features": list(df_cluster_ready.columns),
    "axis_feature_descriptions": {
        "Activity_Score": "거래 횟수와 거래 금액을 통합한 카드 사용 활성도",
        "Decline_Risk_Score": "거래 횟수 변화율과 거래 금액 변화율 감소를 반영한 사용 감소 위험",
        "Inactivity_Score": "최근 12개월 중 비활성 개월 수 기반 비활성 신호",
        "Relationship_Strength_Score": "카드 이용 기간과 관계 상품 수를 통합한 관계 강도",
        "Contact_Burden_Score": "최근 12개월 고객 접촉 횟수 기반 접촉 부담 또는 불만 가능성",
        "Credit_Capacity_Score": "신용한도 규모",
        "Credit_Utilization_Score": "평균 한도 사용률"
    },
    "important_notes": [
        "Attrition_Flag는 clustering input에 사용하지 않고 profiling에서만 사용한다.",
        "Label-encoded categorical variables는 clustering input에 사용하지 않는다.",
        "High_Risk_Signal_Count는 clustering input이 아니라 사후 검증 및 segment profiling에 사용한다.",
        "component-level standardization 후 axis score를 만들고, axis-level standardization을 통해 최종 clustering input을 생성한다."
    ],
    "risk_signal_columns": risk_signal_cols
}

save_json(feature_info, metadata_path)

validation_summary = pd.DataFrame([
    {"item": "input_file", "value": str(input_path.relative_to(ROOT))},
    {"item": "n_rows", "value": len(df_cluster_save)},
    {"item": "n_cluster_features", "value": len(df_cluster_ready.columns)},
    {"item": "n_profile_columns", "value": len(df_profile_save.columns)},
    {"item": "cluster_missing_values", "value": int(df_cluster_save.isnull().sum().sum())},
    {"item": "profile_missing_values", "value": int(df_profile_save.isnull().sum().sum())},
    {"item": "duplicate_rows_original", "value": int(df.duplicated().sum())},
])

validation_summary.to_csv(validation_path, index=False, encoding="utf-8-sig")

print("Saved files:")
for p in [
    cluster_path,
    profile_path,
    axis_path,
    component_path,
    thresholds_path,
    metadata_path,
    validation_path,
    scaler_path,
]:
    print("-", p.relative_to(ROOT))

display(validation_summary)

## 10. Step 3.5 완료 확인

이 단계가 끝나면 다음 파일들이 생성되어야 한다.

- `data/processed/BankChurners_cluster_input.csv`
- `data/processed/BankChurners_profile_table.csv`
- `data/processed/BankChurners_axis_features.csv`
- `data/processed/BankChurners_component_features.csv`
- `data/processed/cluster_feature_info.json`
- `data/processed/preprocessing_validation_summary.csv`
- `data/processed/preprocessing_thresholds.json`
- `data/processed/preprocessing_scalers.pkl`

다음 단계는 `04_clustering_model_selection.ipynb`에서 여러 clustering 알고리즘과 cluster 수를 비교하는 것이다.

In [ ]:
# ============================================================
# 10. Output Check
# ============================================================

expected_outputs = [
    cluster_path,
    profile_path,
    axis_path,
    component_path,
    thresholds_path,
    metadata_path,
    validation_path,
    scaler_path,
]

missing_outputs = [str(p.relative_to(ROOT)) for p in expected_outputs if not p.exists()]

if missing_outputs:
    raise FileNotFoundError(f"생성되지 않은 파일이 있습니다: {missing_outputs}")

print("Step 3.5 completed successfully.")
print("All expected output files exist.")